# Coca-Cola Inventory & Distribution — Python Analytics
Source: supplied CSV (unaltered). All outputs generated only when cells execute. The dataset source/provenance must be confirmed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
p=Path("../data/CocaCola_Inventory_Distribution_MySQL_Ready.csv")
df=pd.read_csv(p)
df["Date"]=pd.to_datetime(df["Date"], errors="coerce")
print(df.shape, df.Date.min(), df.Date.max())
display(df.head())

In [ ]:
print("Missing:", int(df.isna().sum().sum()), "duplicates:", int(df.duplicated().sum()))
display(df.dtypes.astype(str).to_frame("dtype"))
display(df.describe(include="all").T)

In [ ]:
print("Stockout rows:", int(df.Stockout_Flag.sum()))
print("All fill rates 100%:", bool(df.Fill_Rate_Percent.eq(100).all()))
print("All demand = units sold:", bool(df.Demand_Units.eq(df.Units_Sold).all()))
print("Inventory reconciliation mismatches:", int((df.Opening_Inventory_Units+df.Production_Units-df.Units_Sold-df.Waste_Units != df.Closing_Inventory_Units).sum()))
print("Forecast MAE (units/record):", round((df.Forecast_Demand_Units-df.Demand_Units).abs().mean(),2))

In [ ]:
by_market=df.groupby("Market").agg(revenue_usd=("Sales_Revenue_USD","sum"), demand=("Demand_Units","sum"),overstock_rate=("Overstock_Flag","mean")).sort_values("revenue_usd")
ax=by_market.revenue_usd.plot.barh(title="Sales revenue by market",figsize=(10,5));ax.set_xlabel("USD");plt.tight_layout();plt.show()

In [ ]:
monthly=df.assign(month=df.Date.dt.to_period("M")).groupby("month")[["Demand_Units","Forecast_Demand_Units"]].sum()
monthly.index=monthly.index.astype(str)
monthly.plot(figsize=(11,4),title="Monthly recorded demand and forecast");plt.tight_layout();plt.show()

In [ ]:
modes=df.groupby("Transport_Mode").agg(lead_days=("Lead_Time_Days","mean"),avg_cost_usd=("Transport_Cost_USD","mean"),avg_distance_km=("Distance_KM","mean"))
display(modes)
modes.lead_days.plot.bar(title="Mean lead time by mode — unadjusted",figsize=(8,4));plt.tight_layout();plt.show()

In [ ]:
by_market.overstock_rate.mul(100).plot.bar(title="Overstock-flag share of records by market (%)",figsize=(10,4));plt.tight_layout();plt.show()

## Interpretation / limitations
- These are descriptive associations, not causal effects.
- Stockout_Flag and Fill_Rate_Percent have no variation.
- Source/provenance and definitions of flags/turnover must be verified.
- No original values were changed; parsing Date is an in-memory analysis conversion.
- Where inventory-balance assumptions fail, investigate before operational use.